## Setup

First, let's set up the Python environment and import necessary libraries.

In [2]:
import sys
from pathlib import Path

# Get the BICEP root directory (three levels up from this notebook)
bicep_root = Path.cwd().parent.parent.parent
if str(bicep_root) not in sys.path:
    sys.path.insert(0, str(bicep_root))

import pandas as pd
import numpy as np

print("Environment setup complete!")

Environment setup complete!


## Scenario Overview

### BAU (Business As Usual)
Reflects current policies and trends without aggressive electrification policies.
- Moderate technology adoption rates
- Lower overall electrical load growth

### High
Assumes aggressive decarbonization policies and electrification.
- Higher technology adoption rates
- Significant increase in electrical load
- Greater infrastructure upgrade requirements

Let's run analyses for both scenarios and compare the results.

In [3]:
# Load pre-computed results from CSV files
data_path = bicep_root / 'data' / 'parsed_inputs'

print("Loading BAU scenario results...")
bau_results = pd.read_csv(data_path / 'bicep_results_bau_all_states.csv')

print("Loading High scenario results...")
high_results = pd.read_csv(data_path / 'bicep_results_high_all_states.csv')

print(f"\nScenarios loaded successfully!")
print(f"  BAU: {len(bau_results):,} buildings")
print(f"  High: {len(high_results):,} buildings")

Loading BAU scenario results...
Loading High scenario results...

Scenarios loaded successfully!
  BAU: 884,596 buildings
  High: 884,596 buildings


## Total Cost Comparison

In [7]:
# Calculate total costs (using weighted_cost which reflects infrastructure upgrade costs)
bau_total = bau_results['weighted_cost'].sum()
high_total = high_results['weighted_cost'].sum()
difference = high_total - bau_total
percent_increase = (difference / bau_total) * 100

# Residential vs Commercial
bau_res = bau_results[bau_results['residential'] == 1]['weighted_cost'].sum()
bau_com = bau_results[bau_results['residential'] == 0]['weighted_cost'].sum()
high_res = high_results[high_results['residential'] == 1]['weighted_cost'].sum()
high_com = high_results[high_results['residential'] == 0]['weighted_cost'].sum()

print("="*70)
print("TOTAL ANNUALIZED INFRASTRUCTURE UPGRADE COSTS (2020-2050)")
print("="*70)
print(f"\nBAU Scenario:")
print(f"  Total:           ${bau_total:>20,.0f}")
print(f"  Residential:     ${bau_res:>20,.0f}")
print(f"  Commercial:      ${bau_com:>20,.0f}")

print(f"\nHigh Scenario:")
print(f"  Total:           ${high_total:>20,.0f}")
print(f"  Residential:     ${high_res:>20,.0f}")
print(f"  Commercial:      ${high_com:>20,.0f}")

print(f"\nDifference (High - BAU):")
print(f"  Absolute:        ${difference:>20,.0f}")
print(f"  Percent:         {percent_increase:>20.1f}%")
print("="*70)

TOTAL ANNUALIZED INFRASTRUCTURE UPGRADE COSTS (2020-2050)

BAU Scenario:
  Total:           $       5,709,905,030
  Residential:     $       5,167,163,643
  Commercial:      $         542,741,387

High Scenario:
  Total:           $       7,573,495,293
  Residential:     $       7,042,170,162
  Commercial:      $         531,325,131

Difference (High - BAU):
  Absolute:        $       1,863,590,263
  Percent:                         32.6%


## Cost Distribution by Building Type

In [8]:
# Create comparison chart (text-based to avoid rendering issues)
scenarios = ['BAU', 'High']
residential_costs = [bau_res, high_res]
commercial_costs = [bau_com, high_com]

print("\nComparison by Building Type:")
print(f"{'Scenario':<10} {'Residential':>20} {'Commercial':>20} {'Total':>20}")
print("-" * 72)
for i, scenario in enumerate(scenarios):
    total = residential_costs[i] + commercial_costs[i]
    print(f"{scenario:<10} ${residential_costs[i]:>18,.0f} ${commercial_costs[i]:>18,.0f} ${total:>18,.0f}")


Comparison by Building Type:
Scenario            Residential           Commercial                Total
------------------------------------------------------------------------
BAU        $     5,167,163,643 $       542,741,387 $     5,709,905,030
High       $     7,042,170,162 $       531,325,131 $     7,573,495,293


## Cost Drivers Comparison

Let's visualize how different technologies drive costs in each scenario.

In [13]:
# Analyze cost distribution and drivers
def analyze_cost_distribution(df, scenario_name):
    """Analyze cost distribution and drivers"""
    
    total_buildings = len(df)
    buildings_upgraded = (df['upgrade_required'] == 1).sum()
    
    print(f"\n{scenario_name} - Cost Distribution Analysis:")
    print(f"  Total buildings: {total_buildings:,}")
    print(f"  Buildings requiring upgrades: {buildings_upgraded:,} ({buildings_upgraded/total_buildings*100:.1f}%)")
    print(f"  Total cost: ${df['weighted_cost'].sum():,.0f}")
    print(f"  Mean cost (all): ${df['weighted_cost'].mean():,.0f}")
    print(f"  Mean cost (upgraded only): ${df[df['weighted_cost'] > 0]['weighted_cost'].mean():,.0f}")
    
    # State distribution
    top_states = df.groupby('state')['weighted_cost'].sum().nlargest(5)
    print(f"  Top 5 states by cost:")
    for state, cost in top_states.items():
        print(f"    {state}: ${cost:,.0f}")

analyze_cost_distribution(bau_results, "BAU")
analyze_cost_distribution(high_results, "High")


BAU - Cost Distribution Analysis:
  Total buildings: 884,596
  Buildings requiring upgrades: 118,845 (13.4%)
  Total cost: $5,709,905,030
  Mean cost (all): $48,045
  Mean cost (upgraded only): $49,167
  Top 5 states by cost:
    CA: $989,904,653
    TX: $394,192,027
    MI: $270,999,184
    FL: $266,361,645
    NY: $232,398,898

High - Cost Distribution Analysis:
  Total buildings: 884,596
  Buildings requiring upgrades: 172,829 (19.5%)
  Total cost: $7,573,495,293
  Mean cost (all): $43,821
  Mean cost (upgraded only): $50,292
  Top 5 states by cost:
    CA: $1,096,940,988
    TX: $509,608,501
    MI: $498,461,227
    NY: $422,933,747
    IL: $378,266,081


In [14]:
# Analyze residential vs commercial breakdown
print("\n" + "="*70)
print("BUILDING TYPE BREAKDOWN")
print("="*70)

for scenario_name, df in [("BAU", bau_results), ("High", high_results)]:
    res_count = (df['residential'] == 1).sum()
    com_count = (df['residential'] == 0).sum()
    res_cost = df[df['residential'] == 1]['weighted_cost'].sum()
    com_cost = df[df['residential'] == 0]['weighted_cost'].sum()
    
    print(f"\n{scenario_name}:")
    print(f"  Residential: {res_count:,} buildings, ${res_cost:,.0f} total cost")
    print(f"  Commercial: {com_count:,} buildings, ${com_cost:,.0f} total cost")


BUILDING TYPE BREAKDOWN

BAU:
  Residential: 548,916 buildings, $5,167,163,643 total cost
  Commercial: 335,680 buildings, $542,741,387 total cost

High:
  Residential: 548,916 buildings, $7,042,170,162 total cost
  Commercial: 335,680 buildings, $531,325,131 total cost


## Costs by Year

In [15]:
# Get costs by state (top 10)
bau_by_state = bau_results.groupby('state')['weighted_cost'].sum().sort_values(ascending=False).head(10)
high_by_state = high_results.groupby('state')['weighted_cost'].sum().sort_values(ascending=False).head(10)

# Create comparison - use intersection of states to avoid KeyError
common_states = list(set(bau_by_state.index) & set(high_by_state.index))
comparison_states = pd.DataFrame({
    'State': bau_by_state[common_states].index,
    'BAU': bau_by_state[common_states].values,
    'High': high_by_state[common_states].values
})

comparison_states['Difference'] = comparison_states['High'] - comparison_states['BAU']
comparison_states['Percent Increase'] = (comparison_states['Difference'] / comparison_states['BAU'] * 100).round(1)

print("\nTop 10 States by Total Infrastructure Upgrade Costs:")
print("="*90)
for idx, row in comparison_states.iterrows():
    print(f"{row['State']:<5} BAU: ${row['BAU']:>15,.0f}  |  High: ${row['High']:>15,.0f}  |  Increase: {row['Percent Increase']:>6.1f}%")


Top 10 States by Total Infrastructure Upgrade Costs:
NY    BAU: $    232,398,898  |  High: $    422,933,747  |  Increase:   82.0%
NC    BAU: $    191,439,945  |  High: $    259,295,244  |  Increase:   35.4%
IL    BAU: $    199,236,601  |  High: $    378,266,081  |  Increase:   89.9%
NJ    BAU: $    199,440,424  |  High: $    282,970,833  |  Increase:   41.9%
CA    BAU: $    989,904,653  |  High: $  1,096,940,988  |  Increase:   10.8%
TX    BAU: $    394,192,027  |  High: $    509,608,501  |  Increase:   29.3%
MI    BAU: $    270,999,184  |  High: $    498,461,227  |  Increase:   83.9%
OH    BAU: $    164,604,607  |  High: $    325,192,398  |  Increase:   97.6%
FL    BAU: $    266,361,645  |  High: $    349,711,313  |  Increase:   31.3%


## Costs by State (Top 10)

In [19]:
# Analyze regional variation
print("\nRegional Analysis:")
print("-" * 90)
print(f"California (largest):")
ca_bau = bau_results[bau_results['state'] == 'CA']['weighted_cost'].sum()
ca_high = high_results[high_results['state'] == 'CA']['weighted_cost'].sum()
print(f"  BAU: ${ca_bau:,.0f}")
print(f"  High: ${ca_high:,.0f}")
if ca_bau > 0:
    print(f"  Increase: {((ca_high - ca_bau) / ca_bau * 100):.1f}%")


Regional Analysis:
------------------------------------------------------------------------------------------
California (largest):
  BAU: $989,904,653
  High: $1,096,940,988
  Increase: 10.8%


In [16]:
# Analyze regional variation
print("\nRegional Analysis:")
print("-" * 90)
print(f"California (largest):")
ca_bau = bau_results[bau_results['state'] == 'CA']['weighted_cost'].sum()
ca_high = high_results[high_results['state'] == 'CA']['weighted_cost'].sum()
print(f"  BAU: ${ca_bau:,.0f}")
print(f"  High: ${ca_high:,.0f}")
print(f"  Increase: {((ca_high - ca_bau) / ca_bau * 100):.1f}%")


Regional Analysis:
------------------------------------------------------------------------------------------
California (largest):
  BAU: $989,904,653
  High: $1,096,940,988
  Increase: 10.8%


## Capacity Requirements Comparison

In [17]:
# Compare capacity requirements by analyzing upgrade patterns
print("\n" + "="*70)
print("CAPACITY REQUIREMENT ANALYSIS")
print("="*70)

for scenario_name, df in [("BAU", bau_results), ("High", high_results)]:
    residential = df[df['residential'] == 1]
    
    # Buildings needing upgrades
    upgrades_needed = (residential['upgrade_required'] == 1).sum()
    total_residential = len(residential)
    
    # Capacity metrics (max electrical loads)
    avg_load = residential['max_elec_consumption_kwh'].mean()
    total_load = residential['max_elec_consumption_kwh'].sum()
    
    print(f"\n{scenario_name} - Residential:")
    print(f"  Buildings needing upgrades: {upgrades_needed:,} / {total_residential:,} ({upgrades_needed/total_residential*100:.1f}%)")
    print(f"  Average max electrical load: {avg_load:,.0f} kWh/year")
    print(f"  Total peak load: {total_load/1e6:,.1f} million kWh/year")


CAPACITY REQUIREMENT ANALYSIS

BAU - Residential:
  Buildings needing upgrades: 80,197 / 548,916 (14.6%)
  Average max electrical load: 2 kWh/year
  Total peak load: 1.3 million kWh/year

High - Residential:
  Buildings needing upgrades: 120,938 / 548,916 (22.0%)
  Average max electrical load: 2 kWh/year
  Total peak load: 1.3 million kWh/year


## Key Findings

### Cost Differences
The High scenario requires significantly higher infrastructure investments due to increased technology adoption.

In [18]:
# Summary findings
print("\n" + "="*70)
print("KEY FINDINGS: SCENARIO COMPARISON")
print("="*70)

print(f"\n1. COST IMPACT:")
print(f"   The High scenario requires ${difference:,.0f} more in annual costs")
print(f"   This represents a {percent_increase:.1f}% increase over BAU")

print(f"\n2. RESIDENTIAL VS COMMERCIAL:")
res_increase = ((high_res - bau_res) / bau_res) * 100
com_increase = ((high_com - bau_com) / bau_com) * 100
print(f"   Residential increase:  {res_increase:.1f}%")
print(f"   Commercial increase:   {com_increase:.1f}%")

print(f"\n3. BUILDING COVERAGE:")
bau_upgraded = (bau_results['upgrade_required'] == 1).sum()
high_upgraded = (high_results['upgrade_required'] == 1).sum()
print(f"   BAU - Buildings requiring upgrades: {bau_upgraded:,} ({bau_upgraded/len(bau_results)*100:.1f}%)")
print(f"   High - Buildings requiring upgrades: {high_upgraded:,} ({high_upgraded/len(high_results)*100:.1f}%)")

print(f"\n4. REGIONAL VARIATION:")
print(f"   California shows the highest costs in both scenarios")
ca_pct_bau = ca_bau / bau_total * 100
ca_pct_high = ca_high / high_total * 100
print(f"   CA represents {ca_pct_bau:.1f}% of BAU costs and {ca_pct_high:.1f}% of High costs")

print("\n" + "="*70)


KEY FINDINGS: SCENARIO COMPARISON

1. COST IMPACT:
   The High scenario requires $1,863,590,263 more in annual costs
   This represents a 32.6% increase over BAU

2. RESIDENTIAL VS COMMERCIAL:
   Residential increase:  36.3%
   Commercial increase:   -2.1%

3. BUILDING COVERAGE:
   BAU - Buildings requiring upgrades: 118,845 (13.4%)
   High - Buildings requiring upgrades: 172,829 (19.5%)

4. REGIONAL VARIATION:
   California shows the highest costs in both scenarios
   CA represents 17.3% of BAU costs and 14.5% of High costs



## Implications

### Policy Considerations

1. **Infrastructure Planning**: High scenario requires significantly more grid infrastructure investment
2. **Cost Distribution**: Impacts fall unevenly across regions
3. **Timeline**: Early action is needed to spread costs and avoid peak period congestion
4. **Technology Mix**: Different technologies have different infrastructure requirements

### Next Steps

- Review the [Data Requirements](data-requirements.html) notebook to understand technology adoption patterns
- Explore the [Custom Distributions](custom-distributions.html) notebook to learn about cost variations
- Check the [API Reference](../api-reference.html) for custom analysis methods